# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Multinomial Naive Bayes baseline
- train Linear SVM
- run formal model comparison and promote the best flat classifier
- train hierarchical field -> subfield Linear SVM
- run NMF topic modeling and evaluation
- zip outputs for download

It does **not** collect data from APIs or repositories.


## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

Repo: https://github.com/krish-anu/researchlanka-ai.git
Branch: main
Code dir: /kaggle/working/code
Backend dir: /kaggle/working/code/backend
Dataset data dir: /kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Output zip: /kaggle/working/researchlanka-kaggle-outputs.zip


## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [2]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

total 12
drwxr-xr-x 3 root root 4096 Aug 30 13:45 .
drwxr-xr-x 8 root root 4096 Aug 30 13:45 ..
drwxr-xr-x 3 root root 4096 Aug 30 13:45 datasets
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/anusankrishnathas
/kaggle/input/datasets/anusankrishnathas/raw-data1
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Dataset path OK


## 3. Clone Or Pull Latest Main Branch

In [3]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

/kaggle/working
Cloning into 'code'...
remote: Enumerating objects: 3424, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 3424 (delta 58), reused 55 (delta 49), pack-reused 3197 (from 2)
Receiving objects: 100% (3424/3424), 13.77 MiB | 14.17 MiB/s, done.
Resolving deltas: 100% (2042/2042), done.
/kaggle/working/code
6b5aca3 (HEAD -> main, origin/main, origin/feature/machine-learning, origin/HEAD) Merge pull request #484 from krish-anu/feature/machine-learning
d181985 Merge branch 'feature/machine-learning' of https://github.com/krish-anu/researchlanka-ai into feature/machine-learning
64fd2bb Add Linear SVM stop word regression tests
backend       docs		 frontend	   Makefile   README.md
CHANGELOG.md  dse-project.ipynb  KAGGLE_README.md  notebooks  scripts


## 4. Copy Uploaded Raw Data Into Backend

In [4]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

/kaggle/working/code/backend
data/processed/repositories/sliit.jsonl
data/processed/repositories/ruh.jsonl
data/processed/repositories/seu.jsonl
data/processed/repositories/nsf.jsonl
data/processed/repositories/cmb.jsonl
data/processed/repositories/busl.jsonl
data/processed/repositories/uom.jsonl
data/processed/repositories/jfn_medicine.jsonl
data/processed/repositories/ou.jsonl
data/processed/repositories/jfn_research.jsonl
data/processed/repositories/pdn.jsonl
data/processed/sljol.csv
data/processed/repositories_combined.csv
data/processed/crossref/crossref_sri_lanka_works.csv
data/processed/crossref/crossref_sri_lanka_works.jsonl
data/reports/dagster_collection_summary_20260804T085133Z.json
data/config/repositories.json
data/raw/vau/oai_dc.jsonl
data/raw/uom/oai_dc.jsonl
data/raw/ruh/oai_dc.jsonl
data/raw/uwu/rest_items.jsonl
data/raw/openalex/openalex_sri_lanka_pagination_audit.json
data/raw/openalex/openalex_sri_lanka_works.parquet
data/raw/openalex/openalex_sri_lanka_works.csv
da

## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [5]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)
!pip install dagster==1.13.16 dagster-webserver
!pip install -e dagster-quickstart
!python -m dagster --version

/kaggle/working/code/backend
ERROR: Invalid requirement: '<': Expected package name at the start of dependency specifier
    <
    ^
INFO: pip is looking at multiple versions of dagster-webserver to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.0/226.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.7 MB/s eta 0:00:00
  

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [6]:
%cd /kaggle/working/code/backend/dagster-quickstart
!python -m dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

/kaggle/working/code/backend/dagster-quickstart
/usr/local/lib/python3.12/dist-packages/click/core.py:853: SupersessionWarning: Function `job_execute_command` is superseded and its usage is discouraged. Use 'dg launch --job <job_name>' instead.
  return callback(*args, **kwargs)

  Telemetry:

  As an open-source project, we collect usage statistics to inform development priorities. For more
  information, read https://docs.dagster.io/about/telemetry.

  We will not see or store any data that is processed by your code.

  To opt-out, add the following to $DAGSTER_HOME/dagster.yaml, creating that file if necessary:

    telemetry:
      enabled: false


  Welcome to Dagster!

  If you have any questions or would like to engage with the Dagster team, please join us on Slack
  (https://bit.ly/39dvSsF).

2026-08-30 13:47:18 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - dc1a6627-a7ab-496f-9d97-8651dfa0d0c4 - 108 - RUN_START - Started execution of run for "researc

## 7. Verify Preprocessing Outputs

In [7]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


/kaggle/working/code/backend
-rw-r--r-- 1 root root 126M Aug 30 13:47 data/processed/repositories_combined.csv
-rw-r--r-- 1 root root 49M Aug 30 13:47 data/processed/sljol.csv
-rw-r--r-- 1 root root 286M Aug 30 14:03 data/processed/common/common_publications_final.csv
-rw-r--r-- 1 root root 377M Aug 30 14:08 data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/common_publications_final.csv rows= 178162 columns= 56
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv rows= 137856 columns= 71


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [8]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

/kaggle/working/code/backend
python scripts/modeling/generate_publication_text_embeddings.py --input data/processed/common/common_publications_final.csv --output data/models/publication_text_embeddings.parquet --model-output data/models/publication_text_embedding_model.joblib --manifest-output data/models/publication_text_embeddings_manifest.json --summary-output data/models/publication_text_embeddings_summary.txt --text-columns title,abstract,topics,keywords,concepts --metadata-columns record_number,publication_year,title,doi,openalex_id,source_dataset,source_institution_id,source_record_id --embedding-dim 512 --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 
Generated publication text embeddings: rows=177257, dim=512, output=data/models/publication_text_embeddings.parquet
-rw------- 1 root root 395M Aug 30 14:15 data/models/publication_text_embedding_model.joblib
-rw-r--r-- 1 root root 526M Aug 30 14:15 data/models/publication_text_embeddings.parquet
-rw------- 1 root roo

## 9. Train Best-Quality Logistic Regression

In [9]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

/kaggle/working/code/backend
python scripts/modeling/train_logistic_regression_classifier.py --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --model-output data/models/logistic_regression_primary_domain.joblib --metrics-output data/models/logistic_regression_primary_domain_metrics.txt --label-counts-output data/models/logistic_regression_primary_domain_labels.csv --predictions-output data/models/logistic_regression_primary_domain_predictions.csv --manifest-output data/models/logistic_regression_primary_domain_manifest.json --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 --min-class-count 20 --test-size 0.2 --max-iter 2000 
Trained logistic_regression classifier on 52,268 rows.
Classes: 4
Accuracy: 0.8705
Balanced accuracy: 0.8656
Macro F1: 0.8620
Model: data/models/logistic_regression_primary_domain.joblib
Model SHA-256: 80aa39170a04fe6cdca45885c9f877db845d23ebfb6b490052fe967b3

In [10]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

/kaggle/working/code/backend
Obtaining file:///kaggle/working/code/backend
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for research-analytics-framework (pyproject.toml) ... done
  Created wheel for research-analytics-framework: filename=research_analytics_framework-0.1.0-0.editable-py3-none-any.whl size=8228 sha256=61ad0ffc187f32c72bc45d84da8450aa6d36322767465c2700f9effe152a172e
  Stored in directory: /tmp/pip-ephem-wheel-cache-le8otuky/wheels/48/6f/98/3205cf08ddaa25af658fb7d96745a6be29b5c675f81653f651
Successfully built research-analytics-framework


## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [11]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000
!cat data/models/linear_svm_primary_domain_metrics.txt

/kaggle/working/code/backend
Trained Linear SVM classifier on 52,268 rows.
Classes: 4
Best C: 1.0
CV macro F1: 0.8728
Accuracy: 0.8870
Macro F1: 0.8772
Model: /kaggle/working/code/backend/data/models/linear_svm_primary_domain.joblib
Model SHA-256: 385ce2836800091593b27ded2a0ba75947f2abd3888f64b50321b42355a3ba6c
Metrics: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_predictions.csv
Manifest: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_manifest.json
Publication Linear SVM classifier

model_family: linear_svm
input_csv: data/processed/common/common_publications_final.csv
label_column: primary_domain
text_columns: title, abstract, topics, keywords, concepts
input_rows: 178162
usable_rows: 52268
train_rows: 44427
test_rows: 7841
class_count: 4
best_C: 1.0
cv_macro_f1: 0.8728
accuracy: 0.8870
macro_f1: 0.8772
weighted_f1: 0.8869

Class distribution:
Physical

## 11. Train Naive Bayes Baseline


In [12]:
%cd /kaggle/working/code/backend
!make train-nb PYTHON=python \
  NB_INPUT=data/processed/common/common_publications_final.csv \
  NB_LABEL_COLUMN=primary_domain \
  NB_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  NB_ALPHA=1.0 \
  NB_MIN_CLASS_COUNT=20 \
  NB_TEST_SIZE=0.2
!cat data/models/multinomial_nb_primary_domain_metrics.txt


/kaggle/working/code/backend
python -m src.modeling.training --model-family multinomial_nb --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --alpha 1.0 --min-class-count 20 --test-size 0.2 
Trained multinomial_nb classifier on 52,268 rows.
Classes: 4
Accuracy: 0.8133
Balanced accuracy: 0.7887
Macro F1: 0.7935
Model: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain.joblib
Model SHA-256: f013eb3d8934fda97f0797182839dfb4341ae5672935fa689c0a4de9220e998e
Metrics: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_predictions.csv
Confusion matrix: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_confusion_matrix.csv
Per-class results: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_per_class.csv
Manifest: /kaggle/work

## 12. Formal Flat Classifier Comparison

This retrains Logistic Regression and Linear SVM with the same data settings, ranks by macro F1, and copies the winner into `data/models/final/`.


In [13]:
%cd /kaggle/working/code/backend
!python scripts/modeling/compare_classification_models.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --ranking-metric macro_f1

!cat data/models/classification_comparison/model_comparison.csv
!ls -lh data/models/final


/kaggle/working/code/backend
Compared 2 classification model families.
Ranking metric: macro_f1
Best model: linear_svm
Comparison: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison.csv
Manifest: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison_manifest.json
Final field model: /kaggle/working/code/backend/data/models/final/publication_field_classifier.joblib
Final manifest: /kaggle/working/code/backend/data/models/final/publication_field_classifier_manifest.json



total 8.2M
-rw------- 1 root root 8.2M Aug 30 14:30 publication_field_classifier.joblib
-rw------- 1 root root  693 Aug 30 14:30 publication_field_classifier_manifest.json
-rw------- 1 root root  993 Aug 30 14:30 publication_field_classifier_metrics.txt


## 13. Evaluate Prediction Files Together


In [14]:
%cd /kaggle/working/code/backend
!make evaluate-models PYTHON=python \
  EVAL_PREDICTIONS="--predictions-csv data/models/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/linear_svm_primary_domain_predictions.csv" \
  EVAL_OUTPUT_DIR=data/models/evaluation
!find data/models/evaluation -maxdepth 2 -type f -print | sort


/kaggle/working/code/backend
python -m src.modeling.evaluation --predictions-csv data/models/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/linear_svm_primary_domain_predictions.csv --output-dir data/models/evaluation 
Evaluation: multinomial_nb_primary_domain

rows: 10454
class_count: 4
accuracy: 0.8133
balanced_accuracy: 0.7887
macro_precision: 0.8015
macro_recall: 0.7887
macro_f1: 0.7935
weighted_f1: 0.8115

Per-class results:
label                        support    prec  recall      f1  most confused with
Physical Sciences               3435   0.843   0.826   0.835  Social Sciences (366)
Social Sciences                 3050   0.807   0.880   0.842  Physical Sciences (218)
Health Sciences                 2509   0.818   0.825   0.822  Social Sciences (173)
Life Sciences                   1460   0.737   0.624   0.676  Health Sciences (245)

Top 10 confusions:
  Physical Scienc

## 14. Train Hierarchical Field -> Subfield Linear SVM


In [15]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_hierarchical.py \
  --input data/processed/common/common_publications_final.csv \
  --field-column primary_field \
  --subfield-column primary_subfield \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-value 1.0 \
  --class-weight balanced \
  --max-iter 5000 \
  --predict-output data/models/linear_svm_hierarchical_predictions.csv
!cat data/models/linear_svm_hierarchical_metrics.txt
!ls -lh data/models/linear_svm_hierarchical*


/kaggle/working/code/backend
Trained hierarchical Linear SVM on 52,268 rows.
Field classes: 26
Subfield models: 26
Field accuracy: 0.8021
Field macro F1: 0.7456
Subfield mean accuracy: 0.9018
Subfield mean macro F1: 0.8136
Field model: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_field.joblib
Subfield models: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_subfield.joblib
Metrics: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_field_metrics.txt
Manifest: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_field_manifest.json
Predictions written to: data/models/linear_svm_hierarchical_predictions.csv
cat: data/models/linear_svm_hierarchical_metrics.txt: No such file or directory
-rw------- 1 root root  25M Aug 30 14:34 data/models/linear_svm_hierarchical_field.joblib
-rw------- 1 root root  693 Aug 30 14:34 data/models/linear_svm_hierarchical_field_labels.csv
-rw------- 1 root root 8.3K Aug 30 14:34 data/models/linear_s

## 15. Run NMF Topic Modeling And Evaluation


In [16]:
%cd /kaggle/working/code/backend
!python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final.csv \
  --output-dir data/processed/common/nmf \
  --k-range 8 10 12 \
  --n-words 15 \
  --naming-words 3 \
  --text-columns title abstract topics keywords concepts
!find data/processed/common/nmf -maxdepth 1 -type f -print | sort
!cat data/processed/common/nmf/nmf_k_sweep_evaluation.csv


/kaggle/working/code/backend
Loading data/processed/common/common_publications_final.csv ...
Shape: (178162, 56)

Cleaning check: 7900 of 178162 rows contain Tamil/Sinhala-script characters; 4414 rows contain metadata-boilerplate phrases (abstract available / editorial / etc.).
clean=True — stripping these at the token/phrase level (rows are never dropped).

No --k given, sweeping k in [8, 10, 12] ...
k=  8  coherence_cv=0.8294  diversity=0.908  redundancy=0.016  recon_err=414.5850
k= 10  coherence_cv=0.8461  diversity=0.900  redundancy=0.014  recon_err=414.2297
/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
k= 12  coherence_cv=0.8164  diversity=0.900  redundancy=0.011  recon_err=413.8634

Best k by coherence: 10
Cleaning report: 7900 of 178162 rows had non-Latin chars, 4414 rows had boilerplate phrases. clean=True
895 rows had text before clea

In [17]:
%cd /kaggle/working/code/backend
import shutil
from pathlib import Path

aliases = {
    "data/models/linear_svm_hierarchical_subfield.joblib": "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_field_metrics.txt": "data/models/linear_svm_hierarchical_metrics.txt",
}

for source, target in aliases.items():
    source_path = Path(source)
    target_path = Path(target)
    if source_path.exists() and not target_path.exists():
        shutil.copy2(source_path, target_path)
        print(f"Aliased {source} -> {target}")

!ls -lh data/models/linear_svm_hierarchical*

/kaggle/working/code/backend
Aliased data/models/linear_svm_hierarchical_subfield.joblib -> data/models/linear_svm_hierarchical_subfields.joblib
Aliased data/models/linear_svm_hierarchical_field_metrics.txt -> data/models/linear_svm_hierarchical_metrics.txt
-rw------- 1 root root  25M Aug 30 14:34 data/models/linear_svm_hierarchical_field.joblib
-rw------- 1 root root  693 Aug 30 14:34 data/models/linear_svm_hierarchical_field_labels.csv
-rw------- 1 root root 8.3K Aug 30 14:34 data/models/linear_svm_hierarchical_field_manifest.json
-rw------- 1 root root 2.7K Aug 30 14:34 data/models/linear_svm_hierarchical_field_metrics.txt
-rw------- 1 root root   42 Aug 30 14:34 data/models/linear_svm_hierarchical_field_predictions.csv
-rw------- 1 root root 2.7K Aug 30 14:34 data/models/linear_svm_hierarchical_metrics.txt
-rw-r--r-- 1 root root 295M Aug 30 14:37 data/models/linear_svm_hierarchical_predictions.csv
-rw------- 1 root root 115M Aug 30 14:34 data/models/linear_svm_hierarchical_subfield

## 16. Verify All Modeling Outputs


In [18]:
%cd /kaggle/working/code/backend
from pathlib import Path

required = [
    "data/models/publication_text_embeddings.parquet",
    "data/models/publication_text_embedding_model.joblib",
    "data/models/logistic_regression_primary_domain.joblib",
    "data/models/logistic_regression_primary_domain_metrics.txt",
    "data/models/multinomial_nb_primary_domain.joblib",
    "data/models/multinomial_nb_primary_domain_metrics.txt",
    "data/models/linear_svm_primary_domain.joblib",
    "data/models/linear_svm_primary_domain_metrics.txt",
    "data/models/classification_comparison/model_comparison.csv",
    "data/models/final/publication_field_classifier.joblib",
    "data/models/linear_svm_hierarchical_field.joblib",
    "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_metrics.txt",
    "data/processed/common/nmf/nmf_k_sweep_evaluation.csv",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing modeling outputs:\n" + "\n".join(missing))

for p in required:
    path = Path(p)
    print(f"OK {p} ({path.stat().st_size / (1024*1024):.2f} MB)")


/kaggle/working/code/backend
OK data/models/publication_text_embeddings.parquet (525.67 MB)
OK data/models/publication_text_embedding_model.joblib (394.91 MB)
OK data/models/logistic_regression_primary_domain.joblib (7.56 MB)
OK data/models/logistic_regression_primary_domain_metrics.txt (0.00 MB)
OK data/models/multinomial_nb_primary_domain.joblib (5.13 MB)
OK data/models/multinomial_nb_primary_domain_metrics.txt (0.00 MB)
OK data/models/linear_svm_primary_domain.joblib (8.15 MB)
OK data/models/linear_svm_primary_domain_metrics.txt (0.00 MB)
OK data/models/classification_comparison/model_comparison.csv (0.00 MB)
OK data/models/final/publication_field_classifier.joblib (8.15 MB)
OK data/models/linear_svm_hierarchical_field.joblib (24.94 MB)
OK data/models/linear_svm_hierarchical_subfields.joblib (114.99 MB)
OK data/models/linear_svm_hierarchical_metrics.txt (0.00 MB)
OK data/processed/common/nmf/nmf_k_sweep_evaluation.csv (0.00 MB)


## 17. Zip Outputs For Download


In [19]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip


/kaggle/working/code/backend
  adding: data/processed/ (stored 0%)
  adding: data/processed/common/ (stored 0%)
  adding: data/processed/common/publication_multivalue_items_2016_2026.csv (deflated 90%)
  adding: data/processed/common/common_publications_final_2016_2026_summary.csv (deflated 50%)
  adding: data/processed/common/publication_count_audit.csv (deflated 71%)
  adding: data/processed/common/publication_references.csv (deflated 85%)
  adding: data/processed/common/common_publications_final_2016_2026_multivalue_normalized.csv (deflated 70%)
  adding: data/processed/common/common_publications_all_records.csv (deflated 74%)
  adding: data/processed/common/common_publications_final_2016_2026.csv (deflated 70%)
  adding: data/processed/common/common_publications_final_2016_2026_multivalue_normalized_details.csv (deflated 42%)
  adding: data/processed/common/common_publications_deduplicated.csv (deflated 74%)
  adding: data/processed/common/nmf/ (stored 0%)
  adding: data/processed/

Download these files from the Kaggle output panel:


In [20]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-kaggle-outputs.zip"))


/kaggle/working/researchlanka-kaggle-outputs.zip

## 18. Optional: Create Models-Only Zip


In [21]:
import os
import zipfile
from IPython.display import FileLink, display

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]
    print("Model files found:", len(model_files))
    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))
display(FileLink(models_zip))


Model files found: 57
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 1095.51


/kaggle/working/researchlanka-models-only.zip